# Loading the Libraries

#### These are the libraries that are required for data handling, visualization, Machine learning, modelling, Evaluation and pre-processing. These libraries make the code faster, more accuarate and less error prone

In [ ]:
# For data handling
import pandas as pd
import numpy as np

# For data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# For ML
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_validate

# For classification tasks
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB

# metrics
from sklearn.metrics import accuracy_score, f1_score, average_precision_score, confusion_matrix, precision_score, roc_auc_score, classification_report, recall_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

# For data pre-processing
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder,OneHotEncoder

# For ignoring warnings
import warnings
warnings.filterwarnings("ignore")

# Read the data

#### The Novotel dataset is loaded from the csv file. The dataset is loaded to explore its structure and content. It helps to verify that the dataset is loaded correctly

In [ ]:
# Load dataset
df = pd.read_csv('/Users/tunde/Desktop/CAPSTONE Project/WA_Fn-UseC_-Telco-Customer-Churn 2.csv')
# print first 5 rows
df.head(5)

#### Quick preview of the first 5 rows of the dataset

In [ ]:
# Print tail 
df.tail()

#### Quick preview of the last 5 rows of the dataset

# Dataset Preview

#### To check the size of the dataset in terms of rows and columns, we looked at the dataset shape and info

In [ ]:
# check dataset shape 
df.shape

#### Dataset has 7043 rows and 21 columns.

In [ ]:
# dataset info check
df.info()

#### Two columns are integers (SeniorCitizen and tenure), one column is a float (MonthlyCharges) and all remaining columns are of the object datatype. Overall, the dataset contains three data types: integer, float, and object.

In [ ]:
# Empty rows under TotalCharges 
df[df['TotalCharges'].str.strip() == ""]

#### A total of 11 rows were identified where the TotalCharges field is empty or blank

# Data Cleaning

#### The dataset is cleaned to ensure consistency and readiness for analysis. This includes handling missing values, correcting data types, encoding variables and removing any duplicates (if any)

In [ ]:
# Convert values in 'TotalCharges' to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Display 'TotalCharges' datatype
df['TotalCharges'].dtype

#### The TotalCharges column is first converted to a numeric (float) data type

In [ ]:
# counts missing values for each column  ( Only columns of Totalcharges have missing or empty data)
print(df.isnull().sum().sort_values(ascending=False))

#### Missing values were checked across all columns and only the TotalCharges column was found to contain missing or empty values.

In [ ]:
# Remove rows with missing values 
df.dropna(inplace=True)

print(df.isnull().sum().sort_values(ascending=False))

#### The rows with missing values in the TotalCharges column were removed.

In [ ]:
# Total duplicate rows  (No duplicate Rows)
if 'customerID' in df.columns:
    df = df.drop_duplicates(subset=[col for col in df.columns if col != 'customerID'])

# View duplicates
duplicates = df[df.duplicated()]
duplicates.head()

#### Duplicate data were checked and no Duplicate Rows were detected 

In [ ]:
# Remove id column 
df = df.drop(columns=['customerID'])

# display top 5 rows
df.head(5)

#### The CustomerID column was removed because it does not contribute to the analysis. It is a unique identifier and not relevant for modeling and excluding it also helps maintain data privacy.

# Label Encoding

#### Label encoding is done in order to convert the Churn column from text (Yes or No) into numbers (1 for yes and 0 for No) so that the machine‑learning models can use it to learn patterns

In [ ]:
# Encode target (churn) - Categorical column to numeric  (
df["Churn"] = (df["Churn"] == "Yes").astype(int)

In [ ]:
# Count values (Counting the number of customers that Churn and those that didn't churn)
#--- Churn Rate Summary ---
#Non-Churn (0): 73.51%
#Churn (1): 26.49%)
churn_rate = df['Churn'].value_counts()
churn_prop = df['Churn'].value_counts(normalize=True)

# Plot
plt.figure(figsize=(6,6))
plt.bar(churn_rate.index.astype(str), churn_rate.values, color=['blue', 'red'])

plt.xlabel("Churn")
plt.ylabel("Count")
plt.title("Customer Churn Distribution")

plt.show()

# Print output (safe access)
print("--- Churn Rate Summary ---")
print(f"Non-Churn (0): {churn_prop.get(0, 0):.2%}")
print(f"Churn (1): {churn_prop.get(1, 0):.2%}")

#### Counting the number of customers that Churn and those that didn't churn
#### --- Churn Rate Summary ---
#### Non-Churn (0): 73.51%
#### Churn (1): 26.49%

In [ ]:
# Cleaned data shape 
print("Cleaned Shape:", df.shape)

#### After cleaning, the dataset now contains 7,010 rows and 20 columns compared to the original 7,043 rows and 21 columns.

In [ ]:
# Columns of the dataset  
df.columns.values

#### The output shows the list of 20 column headers for the cleaned dataset

#### We then look at the basic statistics of the numerical columns including the count, mean, std, min and max, 25%, 50% and 75% percentile. This would help us understand the overall structure of the dataset.

In [ ]:
# Description stats for numerical columns  
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe()

#### ----Descriptive data statistics for churn dataset for tenure, Monthlycharges and Totalcharges----
#### Tenure: The average a customer stayed is 32.5 Months. Half of the customers stayed less than 29 months. STD - Some customers leave early, others stay many years.
#### MonthlyCharges: The average monthly charge is 64.89. Half of the customers paid more than 70. STD - Wide variation in pricing may be due to different service bundles.
#### TotalCharges: The average a customer paid since joining is 2290.35. Many customers have relatively low total charges because they are newer. STD - New customers have low total, while long‑term customers accumulate high totals.

In [ ]:
# Drop CustomerID in Dataset
for col in df.columns:
    if "customer" in col.lower() and "id" in col.lower():
        print("Dropping:", col)
        df = df.drop(columns=[col])


In [ ]:
# Show unique values in all columns except float or int  
for col in df.select_dtypes(exclude=['int64', 'float64']):
    print(col, ":", df[col].unique())

#### This output shows the unique values that will be used for EDA and visualization to check the churn analysis.

# Visualizations

#### To further explore patterns and relationships in the dataset, visualizations such as graphs and plots are created for different variables. These help identify customers who churn and uncover the key factors driving churn.

In [ ]:
# Define features and target
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [ ]:
print(df.columns.tolist())


## Gender Vs Churn

#### To explore the churners and non-churners based on gender, we look at a stacked bar chart of churn vs non-churn based on female and male

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['gender'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("Gender vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("Gender")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT corner (inside the chart)
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
# moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()



#### Gender: Churn rates are almost identical for males and females, showing gender has no meaningful impact on churn.

## Dependant vs churn

#### To explore the churners and non-churners based on partner, we look at a stacked bar chart of churn vs non-churn based on customers with partners and those without partners

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['Partner'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("Partner vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("Partner")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT corner (inside chart)
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Partner: Customers without a partner churn more, suggesting single customers may be less loyal.

## Dependant vs Churn

#### To explore the churners and non-churners based on dependant, we look at a stacked bar chart of churn vs non-churn based on if customers have dependant or not 

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['Dependents'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True,ax=ax,color=['blue','red'])

ax.set_title("Dependents vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("Dependents")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT corner (inside chart)
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Dependant: Customers without dependents churn significantly more, indicating that lower household responsibility may relate to higher churn.

## PhoneService vs Churn

#### To explore the churners and non-churners based on phoneservice, we look at a stacked bar chart of churn vs non-churn based on if they have a phonservice or not

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['PhoneService'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("PhoneService vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("PhoneService")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT corner (inside chart)
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)
# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Phone Service: Churn rates are similar across groups, meaning phone service alone does not influence churn.

## Multiple Lines vs Churn

#### To explore the churners and non-churners based on multiple lines, we look at a stacked bar chart of churn vs non-churn based on if they have multiple lines or not

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['MultipleLines'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("MultipleLines vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("MultipleLines")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT corner (inside chart)
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)
# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Multiple Lines: Customers with multiple lines churn more, showing that no multi‑line users are more stable. This could likely be due to higher cost and the complexity of managing multiple lines (family plan)

## Internet service vs Churn

#### To explore the churners and non-churners based on their internet service, we look at a stacked bar chart of churn vs non-churn based on if they have internet service (Fiber optic or DSL) or no internet service at all

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("InternetService vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("InternetService")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT but slightly OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Internet Service: Fiber‑optic customers have the highest churn, while DSL and no‑internet customers churn less with the no-internet  service customers being the least churners. This could be due to higher pricing of fiber plans which has higher speed and reliability. This could also make customers have a higher expectation and when there is a network downtime, customers can get dissatisfied and tend to churn

## Online Security vs Churn

#### To explore the churners and non-churners based on online security, we look at a stacked bar chart of churn vs non-churn based on if they have online security or not or not even internet service to access online security

In [ ]:
#Online Security vs Churn

# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['OnlineSecurity'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("OnlineSecurity vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("OnlineSecurity")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT but OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Online Security: Customers without online security churn much more, making it a strong churn predictor. While those with no internet to even access the online security churn least and are the most stable customers. This could be due higher risk exposure to malware and phishing, leading to loss of trust and then customer churning 

## Online Backup vs Churn

#### To explore the churners and non-churners based on online backup, we make a stacked bar chart of churn vs non-churn based on if customers have online backup or not or not even having internet service for online backup

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['OnlineBackup'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("OnlineBackup vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("OnlineBackup")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT but OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)
# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
         ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Online Backup: Customers without backup services show higher churn, though the effect is moderate.

## Device Protection vs Churn

#### To explore the churners and non-churners based on having device protection, we make a stacked bar chart of churn vs non-churn based on if customers have device protection add-on or not or even no internet for device protection

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['DeviceProtection'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("DeviceProtection vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("DeviceProtection")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT but OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
         ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Device Protection: Lack of device protection correlates with higher churn, similar to other add‑on services. This could be because the customer will be responsible for any damage or fault to the device, so customer churns to a service that provides device protection

## Tech support vs Churn

#### To explore the churners and non-churners based on Tech support, we make a stacked bar chart of churn vs non-churn based on if customers have tech support or not or no tech support for non internet customers

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['TechSupport'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("TechSupport vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("TechSupport")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT but OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
       ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Tech Support: Customers without tech support churn at very high rates, making this one of the strongest churn indicators. Customers with no internet and tech support are most stable. This could be due to a weaker customer relationship with the company as there are no engagements between the two parties

## Straming TV vs Churn

#### To explore the churners and non-churners based on Tech support, we make a stacked bar chart of churn vs non-churn based on if they have streaming service or not or not internet to even stream tv

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['StreamingTV'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("StreamingTV vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("StreamingTV")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT but OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
         ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### StreamingTV: Streaming services show small differences, meaning they are not major churn drivers. Most customers often rely on external streaming platforms and so there is a reduced importance of the streamtv provided by the telecom companies

## Contract type vs Churn

#### To explore the churners and non-churners based on contract type, we make a stacked bar chart of churn vs non-churn based on if customers are on a month to month, yearly or two-year plan

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("Contract vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("Contract")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT but OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Contract Type: Month‑to‑month customers churn far more than 1‑year or 2‑year contract customers. This is the strongest churn factor. Customers with no commitment tend to churn easily since there would be no penalties and they tend to be least loyal. They also pay the most price co pared to the the customers with yearly or two-year contracts

## Paperless billing vs Churn

#### To explore the churners and non-churners based on paperless billing, we make a stacked bar chart of churn vs non-churn based on if customers have a paperless billing or not

In [ ]:
# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['PaperlessBilling'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(7,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("PaperlessBilling vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("PaperlessBilling")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

# Legend in TOP‑RIGHT but OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')
plt.tight_layout()
plt.show()


#### Paperless Billing: Paperless billing customers churn more, likely due to the flexible mindset of the customers as all things are done online including switching to other providers and taking advantage of promotions. Also, these customers are frequently exposed to their monthly charges (which is high), thereby leading to cancellations and churning. 

## Payment Method vs Churn

#### To explore the churners and non-churners based on payment method, we make a stacked bar chart of churn vs non-churn based on if customers have an automatic payment (bank transfer and credit card), e-check or mailed check

In [ ]:

# Create normalized table (row-wise percentages)
prop = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100

# Plot
fig, ax = plt.subplots(figsize=(8,5))
prop.plot(kind='bar',stacked=True, ax=ax,color=['blue','red'])

ax.set_title("PaymentMethod vs Churn (Percentage)")
ax.set_ylabel("Percentage (%)")
ax.set_xlabel("PaymentMethod")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Legend in TOP‑RIGHT but OUTSIDE the bar area
ax.legend(title="Churn",labels=["No (0)", "Yes (1)"],loc='upper left',bbox_to_anchor=(1.02, 1),  
          # moves legend OUTSIDE the bars
    frameon=True)

# Add percentage labels inside bars
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_xy()
    if height > 0:
        ax.text(x + width/2,y + height/2, f"{height:.1f}%", ha='center',va='center',fontsize=9,color='white')

plt.tight_layout()
plt.show()


#### Payment Method: Electronic check users exhibit the highest churn,while customers using automatic payments tend to stay longer This could be due to the lack of automation as e-checks would usually require an active action taken every months when customers get their bills. The mailed check group would likely be older generations who are used to the routine or mailing and so they churn less due to thier payment method

## Churn status based on Total Charges, Monthly charges and Tenure

#### Next, we explore the relationship between the numerical features and churn (using Kernel Density Estimation). This would help show how values are distributed across churned and non-churn customers. KDE shows a smooth distribution and helps capture non-linear patterns

In [ ]:
# Numeric distributions by churn

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, num_cols):
    for label, grp in df.groupby("Churn"):
        
        sns.kdeplot(data=grp, x=col, ax=ax, label=["No Churn", "Churn"][label], linewidth=2,  clip=(0, None) 
                    # prevents negative values from KDE smoothing
                )  # <-- THIS closing parenthesis was missing

    ax.set_title(f"Distribution of {col}", fontsize=12)
    ax.set_xlabel(col)
    ax.legend()

plt.suptitle("Numeric Feature Distributions by Churn", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot_numeric_dist.png")
plt.show()



#### Total Charges: Both churned and non-churned customers are concentrated at lower TotalCharges and this is due to lower tenure by churners as customers who stay longer accumulate more total charges. TotalCharges alone is not a strong independent predictor of churn
#### Monthly Charges: Churned customers tend to peak at higher monthly charges (80 - 90) while non-churn customers mostly have lower charges (20). This shows that churn customers have higher monthly charges
#### Tenure: Churned customers are heavily concentrated at low tenure while non-churn customers are kind of spread out and peak at higher tenure. This shows that customers are most likely to churn early on in their tenure

# Correlation Heatmap for Categorical variables

#### To show how the categorical variables relate to eachother in the dataset, a correlation heatmap of all the features is used. Correlation is shown based on numbers and colors. The more the red color, the higher the relationship and the more the blue color, the lower the relationship

In [ ]:
# Correlation heatmap (All features)
# Most features show weak correlations, meaning no strong linear relationships dominate the dataset.
# Tenure has a strong positive correlation with TotalCharges, because longer‑staying customers accumulate higher charges.
# MonthlyCharges has only a mild relationship with other variables, indicating pricing varies independently.
# Churn shows only weak linear correlations, confirming churn is driven by nonlinear interactions rather than simple one‑to‑one relationships.
# Overall, the heatmap suggests that tree‑based models (Random Forest, XGBoost) are appropriate because they capture complex, nonlinear patterns.

df_corr = df.copy()
for c in df_corr.select_dtypes("object").columns:
    df_corr[c] = LabelEncoder().fit_transform(df_corr[c])
corr = df_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(14,10))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax, annot_kws={"size":7})
ax.set_title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plot_heatmap.png")
plt.show()


#### ---Strong Negative Relationship with churn---
##### Contract and tenure have the strongest negative relationship with churn (-0.40) and (-0.35) resp. showing longer contracts and tenure lowers churn

#### ---Moderate Negative Relationship with churn---
##### OnlineSecurity (-0.29) and TechSupport(-0.28) showing that having these add-ons reduce churn

#### ---Positive Relationship with churn---
##### PaperlessBilling (0.19) and MonthlyCharges (0.19) both show a positive relationship with churn i.e increase in MonthlyCharges increases churn

#### ---Strong relationship with features---
##### TotalCharge and Tenure (0.83) shows the longer the tenure, the higher the totalCharges
##### Monthlycharge and TotalCharge (0.65) shows customers with high MonthlyCharges accumulate higher totalCharges (due to tenure)
##### Contract and Tenure (0.68) shows that Long contracts leads to long tenure

##### Churn is driven by contract, tenure, online security, Tech support and monthly charges

# Correlation Heatmap for Numerical variables

#### To show how the numerical variables relate to eachother in the dataset, a correlation heatmap of all the features is used. Correlation is shown based on numbers and colors. The more the red color, the higher the relationship and the more the blue color, the lower the relationship

In [ ]:
# Correlation heatmap (numerical features)
plt.figure(figsize=(10,6))
corr = df[num_cols + ['Churn']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()


#### Tenure and TotalCharges show a moderate-weak negative correlation with churn. Higher tenure and TotalCharges reduces churn
#### MonthlyCharges has a weak positive relationship with churn. Higher Monthlycharges slightly increases churn
#### Tenure is the most important numerical factor for churn. MonthlyCharges influences churn slightly

# Outlier Check

#### To check for extreme values, outliers and distribution spread in the monthlyCharges, we make a box plot

In [ ]:
# Outlier detection 

plt.figure(figsize=(6,4))

sns.boxplot(x='Churn', y='MonthlyCharges', data=df)

plt.title("Monthly Charges vs Churn")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")

plt.show()

#### Churned customers generally have higher monthly charges than non‑churned customers.
#### The median MonthlyCharges is higher for the churn group, increasing churn risk.
#### Non-churn customers are more spread out with many in low charges region (20–60)
#### There is still significant overlap between the two groups, so monthlCharges alone does not fully determine churn
#### No outliers were detected

# Data Preprocessing - Encoding

####  All the categorical features are converted to numerical features which is to be used by the machine learning model

In [ ]:
# Encoding (One-Hot) - Convert categorical column to numeric
df_encoded = pd.get_dummies(df)
df_encoded = df_encoded.astype(int)

print("Encoded Shape:", df_encoded.shape)

In [ ]:
df_encoded.head()

#### Preview of the first 5 rows of the encoded dataset now showing 46 columns after encoding

In [ ]:
# Define features (X) and target (y)
df = pd.get_dummies(df, drop_first=True)

X = df.drop('Churn', axis=1)
y = df['Churn']

## -- DESIGN --
### Since we have explored the features of the cleaned dataset and identified some trends and patterns, we will now find the best classification model to train the dataset and make predictions of customers that will churn. This model will learn the dataset and be used to evaluate unseen sets of dataset (our test set and other dataset after deployment). But first, we will have to split the dataset into training and test set just to prevent data leakages (model not exposed to the test set) during the model training process. 80% of the dataset will be used for training while 20% will be used for test. We will then scale the numerical features to ensure that they are on a comparable scale (no numerical feature is overpowering another based on their values) and then we can now define the different classification methods to be used for the modelling. Next will be to carryout a 10-fold cross validation of the training set as this is ensure that our model generalizes well and does not overfit. From the results (evaluation metrics) of the robust 10-fold cross validation done, we will choose the metric that best aligns with our project goal which is to identify churn customers and that will be Recall, which measures how well a model identifies actual positives (churners). This would be based on which of the different models produces the highest Recall score. Once the model is identified, we will the look for the best settings of the model which will be better than the default settings used during the 10-fold cv, this process is called hyperparameter tuning. After this, the best precision-recall threshold is also selected based on how much we want to place on our model's power to identify churners (Recall) as opposed to  identifying False positives (Precision), which aligns with the business goal. This is called Threshold tuning. Once we have all the best settings, we will then used our chosen and best performing model together with the best tuned settings to make predictions on the unseen test set. This is a process of evaluating our model. We will then create a confusion matrix which shows how well our model performed in terms of the actual outcome compared to the model's prediction. A precision-Recall curve will also be created to show how we made the trade-off between precision and recall at different thresholds, including the one we selected. Then, the most important features that determines churn are rankes and then the model is saved and can now be loaded to evaluate new sets of unseen dataset upon deployment

# Train Validation and Test set split

#### This divides the dataset into the training and test set. Training set is where the model learns patterns while the test set is where the model is evaluated based on what i learnt. Training set is 80% of all dataset while the test set is 20% of all dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

#### All dataset divided into training set (5608 rows) and test set (1402) all with 30 features. Split is 80% Training and 20% Test

# Feature Scaling 

#### After the Train-Test split, feature scaling is done to adjust the numerical features so that they are on similar scale, making the models learn more effectively

In [ ]:
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train)

# Apply same transformation to test
X_test_scaled = scaler.transform(X_test)

print("Scaled Train shape:", X_train_scaled.shape)
print("Scaled Test shape:", X_test_scaled.shape)

#### Feature scaling was applied using the standardScaler (standardizes numerical features), fitted on only the training data (scaling values, mean & std calculated on training set) while the same scaling rules was applied cosistently to validation and test datasets to prevent data leakage

# Define Models to Use

#### 4 different classification models are selected to be used for this project. There are other classification models out there, but for the sake of this project, we will only be using these 4

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),}

#### All the models to be used are defined

# 10-Fold CV- On Train Set

#### 10-Fold Cross-Validation (CV) on the training evaluates how well your model generalizes while tuning it, without touching the test set. It splits your training data into 10 parts and tests with 10 iterations. Gives a more reliable performance estimate than a single split and reduces risk of overfitting

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = {}

for name, model in models.items():

    acc_list, prec_list, rec_list, f1_list = [], [], [], []
    roc_list, pr_list = [], []

    for train_idx, val_idx in cv.split(X_train, y_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # scale only for Logistic Regression
        if name == "Logistic Regression":
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_val = scaler.transform(X_val)

        # train
        model.fit(X_tr, y_tr)

        # predict
        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]

        # metrics
        acc_list.append(accuracy_score(y_val, y_pred))
        prec_list.append(precision_score(y_val, y_pred, zero_division=0))
        rec_list.append(recall_score(y_val, y_pred))
        f1_list.append(f1_score(y_val, y_pred))
        roc_list.append(roc_auc_score(y_val, y_proba))
        pr_list.append(average_precision_score(y_val, y_proba))

    # store mean metrics
    results[name] = {
        "precision": np.mean(prec_list),
        "recall": np.mean(rec_list),
        "f1": np.mean(f1_list),
        "roc_auc": np.mean(roc_list),
    }

# ---------------------------
# Print results
# ---------------------------
print("\n=== 10-Fold CV Results (Training Set) ===")
for name, scores in results.items():
    print(f"\n{name}")
    for metric, value in scores.items():
        print(f"{metric.upper():10s}: {value:.4f}")

#### Evaluation of the multiple models using a 10-fold cross validation on the training set and shows a summary of evaluation metrics for each model. Recall was used as the primary evaluation metric during cross-validation to prioritize identifying customers likely to churn. This aligns with the business objective of minimizing missed churn cases. Metrics +/- standard deviation shown. Accuracy and AUC can be misleading here

# Model Comparison of CV and Selection

#### After the 10-fold cross-validation, we compare the multiple models using cross-validation results (recall in our case) and choose the best one based on the recall value. Model with highest Recall score is chosen. This aligns with the project objective of identifying churn customers

In [ ]:
# Convert to DataFrame
df_results = pd.DataFrame(results).T

# Sort by Recall (churn priority)
df_results = df_results.sort_values(by="recall", ascending=False)

print("\n=== MODEL COMPARISON (10-Fold CV) ===")
print(df_results)

best_model_name = df_results["recall"].idxmax()
print("\nBest model based on Recall:", best_model_name)

#### This provides the summary of the cross-validation results into a structured table, based on ranking the models by performance. Logistic Regression achieved the highest recall (0.545787) and demonstrated better generalization compared to other models. 

# Hyperparameter Tuning

#### After choosing the best model, we find the best setting for the ML model chosen (Logistic Regression) in order to improve it's performance as opposed to the default settings

In [ ]:
param_grid = [
    {"solver": ["lbfgs"], "C": [0.01, 0.1, 1, 10], "class_weight": [None, "balanced"]},
    {"solver": ["liblinear"], "C": [0.01, 0.1, 1, 10], "class_weight": [None, "balanced"]}
]

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=10,
    scoring="recall",
    n_jobs=-1
)

# Fit on training data
grid.fit(X_train_scaled, y_train)

# Best model
best_model = grid.best_estimator_

print("Best Params:", grid.best_params_)
print("Best CV Recall:", grid.best_score_)

#### The optimized logistic regression model uses strong regularization (C = 0.01), balanced class weights, and the liblinear solver to address class imbalance. It achieved a cross-validated recall of 0.81 showing an effective identification of positive cases. This prioritizes minimizing missed positives, though it may increase false positives, making it suitable for recall-focused applications

# Threshold Tuning

#### After tuning the hyperparameters, we also tune the threshold which is basically adjusting the decision cutoff inorder to balance precision and recall based on your project goal

In [ ]:
# probabilities from TEST set
y_test_prob = best_model.predict_proba(X_test_scaled)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)

print("Threshold | Recall | Precision | F1")
print("-------------------------------------")

for t in thresholds:
    y_pred = (y_test_prob > t).astype(int)

    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred)

    print(f"{t:.2f}      | {recall:.3f}  | {precision:.3f}   | {f1:.3f}")

In [ ]:
best_threshold = 0.4

#### Different classification thresholds were evaluated using the validation set to optimize recall while maintaining reasonable precision. A threshold of 0.4 provided the best balance and was selected as new threshold.¶

# Final Retraining on Full Training set and Test Evaluation

#### Once the best model is tuned to the best performance settings, we then retrain the model on the full training set (since only 90% of the training set was used to train the model during cv) and then measure how well the model performs on completely unseen data (Test dataset)

In [ ]:
# Convert scaled test set back to DataFrame
feature_names = X_train.columns
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Predict on test set
y_test_prob = best_model.predict_proba(X_test_scaled_df)[:, 1]
y_pred = (y_test_prob > best_threshold).astype(int)

print("\nFINAL TEST RESULTS")
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


#### Threshold tuning was applied to improve recall while controlling false positives. The final model achieved a recall of approximately 0.86 with improved precision and F1-score, providing a better balance between identifying churners and limiting unnecessary interventions. Accuracy of 71% shows model classifies 71% of all customers correctly. Recall of 0.86 shows model correctly identifies 86% of customers who churned (most important metric according to business objective). Precision of 0.47 indicates moderate false positives and F1 of 0.61 shows balanced measure of precision and recall. High recall ensures that potential churn customers are rarely missed, enabling proactive retention strategies while Moderate precision implies that some retention efforts may target non-churn customers, increasing operational cost.

# Confusion Matrix

#### This helps to show how well the model predicted each class of data by comparing predictions vs actual values. It breaks down predictions to show exactly where the model is right or wrong

In [ ]:
cm = confusion_matrix(y_test, y_pred)

# row-wise normalization
cm_row = cm.astype(float) / cm.sum(axis=1, keepdims=True)

sns.heatmap(
    cm_row,
    annot=True,
    fmt=".2%",   # percentage per row
    cmap="Blues"
)

plt.title("Confusion Matrix (Row Normalized %)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

#### For non-churners, the model's prediction is 65% correct and 35% (False positives and False Alarms) in-correct while for churners, the model is 86% correct in prediction and 14% in-correct (False negatives and missed churners)

# Precision-Recall Curve

#### This curve shows the trade-off between precision and recall at different classification thresholds. It shows how well the model finds positives (e.g., churners) without making too many false alarms i.e. the best balance between catching churners and avoiding false alarms

In [ ]:
from sklearn.metrics import precision_recall_curve

y_prob = best_model.predict_proba(X_test)[:,1]

precision, recall, _ = precision_recall_curve(y_test, y_test_prob)

plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

#### This shows how precision (how accurate churn predictions are) changes as recall (how many churners) increases thus helping to choose best threshold. This was used to evaluate model performance under different thresholds, providing insight into the trade-off between identifying churners and minimizing false positives. Low recall → very accurate but misses churners...High recall → catches churners but less accurate...Middle → best balance. For the sake of alignment with our business goal of best identifying churners, Our selected threshold for this project is 0.4 (precision = 0.47 and Recall = 0.86)

# Top Features

#### The most important variables that have the strongest influence on your model’s predictions are highlighted. This helps to improve interpretability and help guide in making business decisions

In [ ]:
from sklearn.inspection import permutation_importance

# Convert scaled test set back to DataFrame
feature_names = X_train.columns
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Permutation importance
perm = permutation_importance(
    best_model,
    X_test_scaled_df,   # <-- FIXED
    y_test,
    scoring="recall",
    n_repeats=10,
    random_state=42
)

feature_imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

print(feature_imp.head(10))

#### Customer commitment (Tenure and contract of 2 years) dominates churn driver, while payment method, total charges and services are less determinants of churn. Company should focus on the low tenure customers (incentives and discounts), promote long-term contracts and offer some free add-on services. AUTO-PAY can be implemented for encoraging automatic pay, thereby making the customers churn less

# Save Final Model

#### The trained ML model (Logistic Regression with best tuned settings) is stored to be re-used on other unseen datasets without retraining the model again

In [ ]:
import joblib

# Save model
joblib.dump(best_model, "final_model.pkl")

print("Model saved as final_model.pkl")

# Loading Model for Future Use

#### The model can be loaded inorder to reuse the trained model to make new predictions instantly

In [ ]:
# Load model
loaded_model = joblib.load("final_model.pkl")

# Test prediction
y_pred = loaded_model.predict(X_test)

print("Model loaded successfully")